# Combining RAG and SQL Agents in One Workflow

Production support agents need **two grounding mechanisms**:

- **RAG** for policy prose (returns, shipping, billing FAQs)
- **SQL** for live tabular facts (order status, counts, aggregates)

A single LLM call cannot reliably do both. A **unified workflow** routes each question to the right specialist, merges evidence, and composes one grounded answer.

```
                    ShopCo Unified Support
                           │
                      classify intent
                           │
         ┌─────────────────┼─────────────────┐
         ▼                 ▼                 ▼
    Policy RAG        SQL Agent          Both paths
    (FAISS docs)      (guarded SELECT)   (parallel)
         │                 │                 │
         └─────────────────┼─────────────────┘
                           ▼
                   evidence merge + citations
                           ▼
                      grounded answer
```

This notebook implements the full **ShopCo Unified Support Workflow** with:

1. FAISS policy retrieval
2. Guarded text-to-SQL execution
3. Intent router (policy / order / mixed / general)
4. Plain-Python and **LangGraph** orchestration variants
5. Unified trace and evaluation harness

Self-contained — no prior notebooks required.

In [ ]:
"""
ShopCo Unified Support — RAG + SQL workflow lab.
"""

import json
import os
import re
import sqlite3
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Literal

import faiss
import numpy as np
from langgraph.graph import END, START, StateGraph
from sklearn.feature_extraction.text import TfidfVectorizer
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_DOCS: list[dict[str, str]] = [
    {"id": "pol-returns-01", "title": "Returns Policy",
     "text": "Customers may return items within 30 days of delivery. Refunds in 5-7 business days after warehouse receipt."},
    {"id": "pol-shipping-02", "title": "Shipping Policy",
     "text": "Free shipping over $50. Tracking emailed when carrier scans the package."},
    {"id": "pol-billing-03", "title": "Billing Policy",
     "text": "Charges appear as SHOPCO* on statements. Disputes within 60 days with order ID."},
    {"id": "faq-tracking-04", "title": "Tracking FAQ",
     "text": "Shipped orders have tracking numbers. Processing orders do not yet have tracking."},
]


def create_shopco_db() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript("""
        CREATE TABLE orders (
            order_id TEXT PRIMARY KEY, customer_email TEXT, status TEXT,
            total_usd REAL, tracking TEXT, created_at TEXT
        );
    """)
    conn.executemany(
        "INSERT INTO orders VALUES (?,?,?,?,?,?)",
        [
            ("ORD-1001", "alice@shopco.com", "shipped", 89.50, "1Z999AA10123456784", "2026-07-01"),
            ("ORD-1002", "bob@shopco.com", "processing", 42.00, None, "2026-07-08"),
            ("ORD-1003", "carol@shopco.com", "delivered", 120.00, "1Z999AA10987654321", "2026-06-15"),
        ],
    )
    conn.commit()
    return conn


DB = create_shopco_db()
print(f"Corpus: {len(POLICY_DOCS)} policies, 3 orders")

---

## 1. Why One Workflow, Two Grounding Paths?

| Question | Wrong approach | Right approach |
|----------|----------------|----------------|
| "Return window?" | SQL on orders table | RAG → `pol-returns-01` |
| "ORD-1001 status?" | RAG over policies | SQL → `orders` row |
| "Return policy + ORD-1002 status" | Pick one | **Both** → merge evidence |
| "Hello!" | Force retrieval | General reply, no tools |

Combining paths in **one orchestrator** gives a single trace, consistent citations, and one user-facing answer.

---

## 2. Policy RAG Layer (FAISS)

In [ ]:
@dataclass
class RagHit:
    doc_id: str
    title: str
    text: str
    score: float


class PolicyRAG:
    def __init__(self, docs: list[dict[str, str]]) -> None:
        self.docs = docs
        texts = [f"{d['title']}. {d['text']}" for d in docs]
        self.vectorizer = TfidfVectorizer(stop_words="english")
        matrix = self.vectorizer.fit_transform(texts).astype("float32").toarray()
        faiss.normalize_L2(matrix)
        self.index = faiss.IndexFlatIP(matrix.shape[1])
        self.index.add(matrix)

    def search(self, query: str, top_k: int = 2) -> list[RagHit]:
        vec = self.vectorizer.transform([query]).astype("float32").toarray()
        faiss.normalize_L2(vec)
        scores, idx = self.index.search(vec, top_k)
        hits: list[RagHit] = []
        for s, i in zip(scores[0], idx[0]):
            if i < 0 or s < 0.05:
                continue
            d = self.docs[int(i)]
            hits.append(RagHit(d["id"], d["title"], d["text"], float(s)))
        return hits


RAG = PolicyRAG(POLICY_DOCS)
print(RAG.search("money back guarantee")[0].doc_id)

---

## 3. Guarded SQL Layer

In [ ]:
FORBIDDEN = re.compile(r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE)\b", re.I)


@dataclass
class SqlResult:
    sql: str
    rows: list[dict[str, Any]]
    blocked: bool = False
    error: str = ""


def plan_sql(question: str) -> str | None:
    m = re.search(r"ORD-[0-9]{4}", question.upper())
    if m:
        return f"SELECT order_id, status, total_usd, tracking FROM orders WHERE order_id = '{m.group(0)}'"
    q = question.lower()
    if "how many" in q and "order" in q:
        if "shipped" in q:
            return "SELECT COUNT(*) AS cnt FROM orders WHERE status = 'shipped'"
        return "SELECT COUNT(*) AS cnt FROM orders"
    return None


def execute_guarded_sql(conn: sqlite3.Connection, sql: str) -> SqlResult:
    cleaned = sql.strip().rstrip(";")
    if FORBIDDEN.search(cleaned) or not re.match(r"^SELECT", cleaned, re.I):
        return SqlResult(sql=sql, rows=[], blocked=True, error="only SELECT allowed")
    if "limit" not in cleaned.lower():
        cleaned += " LIMIT 50"
    try:
        rows = [dict(r) for r in conn.execute(cleaned).fetchall()]
        return SqlResult(sql=cleaned, rows=rows)
    except sqlite3.Error as exc:
        return SqlResult(sql=cleaned, rows=[], blocked=True, error=str(exc))


print(execute_guarded_sql(DB, plan_sql("ORD-1001 status") or ""))

---

## 4. Intent Router

In [ ]:
class SupportIntent(str, Enum):
    POLICY = "policy"
    ORDER = "order"
    MIXED = "mixed"
    GENERAL = "general"


def classify_intent(question: str) -> SupportIntent:
    q = question.lower().strip()
    if q in ("hi", "hello", "thanks", "thank you"):
        return SupportIntent.GENERAL
    has_order = bool(re.search(r"ORD-[0-9]{4}", question.upper())) or "order" in q or "tracking" in q
    has_policy = any(t in q for t in ("return", "refund", "shipping", "billing", "policy", "money back"))
    has_count = "how many" in q
    if has_order and has_policy:
        return SupportIntent.MIXED
    if has_order or has_count:
        return SupportIntent.ORDER
    if has_policy:
        return SupportIntent.POLICY
    return SupportIntent.GENERAL


for q in ["Return policy?", "ORD-1001 status", "Return policy and ORD-1002", "Hello!"]:
    print(f"{q} → {classify_intent(q).value}")

---

## 5. Evidence Bundle — Merge RAG + SQL

In [ ]:
@dataclass
class EvidenceItem:
    source: Literal["rag", "sql"]
    ref: str
    snippet: str


@dataclass
class UnifiedResult:
    question: str
    intent: SupportIntent
    evidence: list[EvidenceItem] = field(default_factory=list)
    answer: str = ""
    trace: list[str] = field(default_factory=list)
    grounded: bool = False


def compose_answer(evidence: list[EvidenceItem]) -> str:
    if not evidence:
        return "I don't have verified ShopCo information for that question."
    parts: list[str] = []
    for e in evidence:
        if e.source == "rag":
            parts.append(f"[{e.ref}] {e.snippet[:120]}")
        else:
            parts.append(f"[SQL: {e.ref}] {e.snippet[:120]}")
    cites = ", ".join(e.ref for e in evidence)
    return " ".join(parts) + f"\n\nSources: {cites}"

---

## 6. Hybrid Coordinator (Plain Python)

In [ ]:
class HybridSupportCoordinator:
    def __init__(self, rag: PolicyRAG, db: sqlite3.Connection) -> None:
        self.rag = rag
        self.db = db

    def run(self, question: str) -> UnifiedResult:
        intent = classify_intent(question)
        result = UnifiedResult(question=question, intent=intent)
        result.trace.append(f"intent:{intent.value}")

        if intent == SupportIntent.GENERAL:
            result.answer = "Hello! Ask about ShopCo orders or policies."
            return result

        if intent in (SupportIntent.POLICY, SupportIntent.MIXED):
            hits = self.rag.search(question)
            for h in hits:
                result.evidence.append(EvidenceItem("rag", h.doc_id, h.text))
            result.trace.append(f"rag:hits={len(hits)}")

        if intent in (SupportIntent.ORDER, SupportIntent.MIXED):
            sql = plan_sql(question)
            if sql:
                sr = execute_guarded_sql(self.db, sql)
                result.trace.append(f"sql:blocked={sr.blocked}")
                if sr.rows:
                    result.evidence.append(EvidenceItem("sql", sr.sql[:40], pretty(sr.rows[0])))

        result.grounded = len(result.evidence) > 0
        result.answer = compose_answer(result.evidence)
        result.trace.append("compose:done")
        return result


coordinator = HybridSupportCoordinator(RAG, DB)

demo = coordinator.run("Return policy and status of ORD-1001")
print("Intent:", demo.intent.value)
print("Evidence:", [(e.source, e.ref) for e in demo.evidence])
print(demo.answer[:200], "...")

---

## 7. LangGraph Unified Workflow

Same logic as explicit graph nodes — visible control flow and checkpoint-ready.

In [ ]:
class GraphState(TypedDict):
    question: str
    intent: str
    evidence: list[dict[str, str]]
    answer: str
    trace: list[str]


def classify_node(state: GraphState) -> dict[str, Any]:
    intent = classify_intent(state["question"])
    return {"intent": intent.value, "trace": state.get("trace", []) + [f"classify:{intent.value}"]}


def rag_node(state: GraphState) -> dict[str, Any]:
    hits = RAG.search(state["question"])
    ev = list(state.get("evidence", []))
    for h in hits:
        ev.append({"source": "rag", "ref": h.doc_id, "snippet": h.text})
    return {"evidence": ev, "trace": state["trace"] + [f"rag:{len(hits)}"]}


def sql_node(state: GraphState) -> dict[str, Any]:
    sql = plan_sql(state["question"])
    ev = list(state.get("evidence", []))
    trace = state["trace"]
    if sql:
        sr = execute_guarded_sql(DB, sql)
        trace = trace + [f"sql:rows={len(sr.rows)}"]
        if sr.rows:
            ev.append({"source": "sql", "ref": sr.sql[:40], "snippet": pretty(sr.rows[0])})
    return {"evidence": ev, "trace": trace}


def compose_node(state: GraphState) -> dict[str, Any]:
    if state["intent"] == "general":
        return {"answer": "Hello! Ask about ShopCo orders or policies.", "trace": state["trace"] + ["compose:general"]}
    items = [EvidenceItem(e["source"], e["ref"], e["snippet"]) for e in state.get("evidence", [])]
    return {"answer": compose_answer(items), "trace": state["trace"] + ["compose:unified"]}


def route_after_classify(state: GraphState) -> str:
    intent = state["intent"]
    if intent == "general":
        return "compose"
    if intent == "policy":
        return "rag"
    if intent == "order":
        return "sql"
    return "both"


builder = StateGraph(GraphState)
builder.add_node("classify", classify_node)
builder.add_node("rag", rag_node)
builder.add_node("sql", sql_node)
builder.add_node("both_rag", rag_node)
builder.add_node("both_sql", sql_node)
builder.add_node("compose", compose_node)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_after_classify, {
    "rag": "rag", "sql": "sql", "both": "both_rag", "compose": "compose",
})
builder.add_edge("rag", "compose")
builder.add_edge("sql", "compose")
builder.add_edge("both_rag", "both_sql")
builder.add_edge("both_sql", "compose")
builder.add_edge("compose", END)

unified_graph = builder.compile()

gout = unified_graph.invoke({"question": "Refund timing and ORD-1002 status", "intent": "", "evidence": [], "answer": "", "trace": []})
print("Trace:", " → ".join(gout["trace"]))
print(gout["answer"][:180], "...")

---

## 8. Single-Source Failure Demos

In [ ]:
q_policy = "What is the return window?"
sql_only = execute_guarded_sql(DB, plan_sql(q_policy) or "SELECT 1")
rag_only = RAG.search(q_policy)

print("Policy question — SQL path:", sql_only.rows, "(no policy prose)")
print("Policy question — RAG path:", rag_only[0].doc_id if rag_only else "none")

q_order = "ORD-1001 tracking"
rag_wrong = RAG.search(q_order)
sql_right = execute_guarded_sql(DB, plan_sql(q_order) or "")
print("\nOrder question — RAG top hit:", rag_wrong[0].title if rag_wrong else "none", "(FAQ not live status)")
print("Order question — SQL:", sql_right.rows[0] if sql_right.rows else "none")

---

## 9. Unified Trace Format

In [ ]:
UNIFIED_LOG: list[dict[str, Any]] = []


def run_and_log(question: str) -> UnifiedResult:
    r = coordinator.run(question)
    UNIFIED_LOG.append({
        "ts": utc_now(), "question": question, "intent": r.intent.value,
        "sources": [e.source for e in r.evidence], "grounded": r.grounded,
    })
    return r


for q in ["How many shipped orders?", "Billing dispute process", "Hi there"]:
    r = run_and_log(q)
    print(f"{q} → intent={r.intent.value} sources={[e.source for e in r.evidence]}")

---

## 10. Scenario Suite

In [ ]:
SCENARIOS = [
    ("Policy only", "How long until refund after return?", {"rag"}, SupportIntent.POLICY),
    ("Order only", "Where is order ORD-1001?", {"sql"}, SupportIntent.ORDER),
    ("Mixed", "Return policy and ORD-1003 status", {"rag", "sql"}, SupportIntent.MIXED),
    ("Count", "How many shipped orders?", {"sql"}, SupportIntent.ORDER),
    ("General", "Thanks!", set(), SupportIntent.GENERAL),
]

for label, q, expected_sources, expected_intent in SCENARIOS:
    r = coordinator.run(q)
    sources = {e.source for e in r.evidence}
    ok_intent = r.intent == expected_intent
    ok_sources = expected_sources <= sources if expected_sources else not sources
    mark = "✓" if ok_intent and ok_sources else "✗"
    print(f"{mark} {label}: intent={r.intent.value} sources={sources}")

---

## 11. Eval Harness

In [ ]:
EVAL: list[dict[str, Any]] = [
    {"q": "30 day return", "must_rag": True, "must_sql": False},
    {"q": "ORD-1002 status", "must_rag": False, "must_sql": True},
    {"q": "Return rules and ORD-1001", "must_rag": True, "must_sql": True},
]

passed = 0
for case in EVAL:
    r = coordinator.run(case["q"])
    has_rag = any(e.source == "rag" for e in r.evidence)
    has_sql = any(e.source == "sql" for e in r.evidence)
    ok = (has_rag == case["must_rag"]) and (has_sql == case["must_sql"])
    passed += int(ok)
    print(f"{'✓' if ok else '✗'} {case['q']} rag={has_rag} sql={has_sql}")
print(f"\nEval: {passed}/{len(EVAL)}")

---

## 12. Orchestration Comparison

| Approach | Pros | Cons |
|----------|------|------|
| **HybridSupportCoordinator** | Simple, testable | Less visible graph |
| **LangGraph unified_graph** | Explicit branches, checkpoints | More boilerplate |
| **Single mega-prompt** | Fast to prototype | Wrong source, no trace |
| **Two separate bots** | Isolation | User gets fragmented answers |

---

## 13. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| RAG for order status | Stale/wrong status | SQL path for ORD-* |
| SQL for policy prose | Empty or wrong rows | RAG path |
| No mixed path | Half-answered questions | `MIXED` intent runs both |
| Unmerged citations | User can't audit | `EvidenceItem` bundle |
| Skip router on "thanks" | Wasted FAISS/SQL | `GENERAL` short-circuit |
| No guardrails on SQL | Injection / DDL | Guarded executor always |

---

## 14. Production Checklist

- [ ] Intent router with policy / order / mixed / general
- [ ] RAG index versioned separately from DB schema
- [ ] SQL guardrails on every execution
- [ ] Unified evidence model with source tags
- [ ] Single composed answer with citations
- [ ] Trace log: intent → rag hits → sql rows → compose
- [ ] Eval set covering mixed questions
- [ ] LangGraph or equivalent for checkpoint/HITL extension

---

## 15. Optional Live LLM Compose Step

Retrieve with RAG/SQL deterministically; optional LLM **only** synthesizes the final answer from evidence.

In [ ]:
def compose_with_optional_llm(result: UnifiedResult) -> str:
    if not USE_LIVE_LLM or not result.evidence:
        return result.answer
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    ctx = "\n".join(f"[{e.source}:{e.ref}] {e.snippet}" for e in result.evidence)
    prompt = f"Answer using ONLY this evidence. Cite sources.\n{ctx}\n\nQuestion: {result.question}"
    return llm.invoke(prompt).content


base = coordinator.run("Return window and ORD-1001 status")
print(compose_with_optional_llm(base)[:150], "...")

---

## 16. Quiz

1. When should RAG run vs SQL in the unified workflow?
2. What is the `MIXED` intent and why does it matter?
3. Why keep evidence as separate `rag` and `sql` items?
4. What does LangGraph add over a plain coordinator class?
5. Why run SQL guardrails even when SQL is planner-generated offline?

---

## 17. Summary

**ShopCo Unified Support** combines FAISS policy RAG and guarded SQL in one workflow:

1. **Classify** → policy / order / mixed / general
2. **Retrieve** → FAISS for prose, SQL for rows (or both)
3. **Merge** → `EvidenceItem` list with source tags
4. **Compose** → single grounded answer with citations

Implement as `HybridSupportCoordinator` or a **LangGraph** with conditional edges — same architecture, different control-plane style. Next: **mixed data source orchestration** across APIs, docs, and warehouses.